# IOAI — 2026 Contest Leopard Cat Reid (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
os.makedirs('data', exist_ok=True)
if not os.listdir('data'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2026-contest-leopard-cat-reid/data.zip', 'd.zip')
    zipfile.ZipFile('d.zip').extractall('data')
print('데이터 준비:', sorted(os.listdir('data'))[:8])
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Leopard Cat Re-ID — 모범답안 (사전학습 ConvNeXt 특징 + 분류기)

이미지가 적고(개체당 몇 장) 클래스가 54개라 처음부터 학습은 과적합된다. **ImageNet 사전학습 ConvNeXt-Tiny**
(torchvision, 대회 허용 가중치)를 **동결 특징추출기**로 써서 768-d 임베딩을 뽑고, 그 위에 로지스틱 회귀를
학습한다(빠르고 안정적). accuracy ≈ 0.4~0.6 (다수결 대비 큰 향상, 참고점수 0.658).

In [ ]:
import pandas as pd, os
train = pd.read_csv("data/train.csv")            # filename, label (0..53)
test_list = pd.read_csv("data/test_list.csv")["filename"].tolist()
print("train", len(train), "| classes", train.label.nunique(), "| test", len(test_list))

In [ ]:
import torch, numpy as np
import torchvision.transforms as T
from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights
from PIL import Image
from sklearn.linear_model import LogisticRegression
dev = "cuda" if torch.cuda.is_available() else "cpu"

w = ConvNeXt_Tiny_Weights.DEFAULT
backbone = convnext_tiny(weights=w).to(dev).eval()
backbone.classifier[2] = torch.nn.Identity()     # 768-d 특징
tf = T.Compose([T.Resize((224,224)), T.ToTensor(),
                T.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])])

@torch.no_grad()
def embed(paths, folder):
    E = []
    for i in range(0, len(paths), 32):
        batch = torch.stack([tf(Image.open(f"data/{folder}/{p}").convert("RGB")) for p in paths[i:i+32]]).to(dev)
        E.append(backbone(batch).cpu().numpy())
    return np.concatenate(E)

Xtr = embed(train.filename.tolist(), "train")
Xte = embed(test_list, "test")
clf = LogisticRegression(max_iter=3000, C=1.0).fit(Xtr, train.label)
pred = clf.predict(Xte)
pd.DataFrame({"filename": test_list, "label": pred.astype(int)}).to_csv("submission.csv", index=False)
print("saved submission.csv", len(test_list))

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)